# ComfyUI Google Cloud TPU v3-8 (8-Core) Parallel Inference Benchmark

This notebook demonstrates native Google Cloud TPU support in ComfyUI (`torch.device('tpu')` via `torch_tpu`), running 8 independent ComfyUI instances across the 8 TPU cores of a Google Cloud / Kaggle TPU v3-8 accelerator, and executing parallel diffusion inference with Qwen-Image 2.1.

In [ ]:
import os
import sys
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
has_acc = hasattr(torch, 'accelerator') and torch.accelerator.is_available()
print(f"torch.accelerator available: {has_acc}")
if has_acc:
    print(f"Current accelerator: {torch.accelerator.current_accelerator()}")
    print(f"Accelerator device count: {torch.accelerator.device_count()}")


In [ ]:
# Install TorchTPU and ComfyUI dependencies
!pip install -q torch_tpu --extra-index-url https://download.pytorch.org/whl/cpu
!pip install -q psutil comfy-aimdo comfy-kitchen torchvision


In [ ]:
# Clone or setup patched ComfyUI with native TPU support
import os
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git ComfyUI

# Ensure requirements are installed
!pip install -q -r ComfyUI/requirements.txt


In [ ]:
# Create directories for models per Neptune Motion specification
!mkdir -p ComfyUI/models/unet ComfyUI/models/clip ComfyUI/models/vae

# 1. Diffusion Model (Qwen-Image 2.1 Turbo Quantized GGUF)
!wget -c -q --show-progress -O ComfyUI/models/unet/qwen-image-2.1-UC-Q4_K_M.gguf \
  "https://huggingface.co/abenzerps/Qwen-Image-2.1-Turbo-Quantized/resolve/main/qwen-image-2.1-turbo-Q4_K_M.gguf"

# 2. Text Encoder CLIP
!wget -c -q --show-progress -O ComfyUI/models/clip/qwen3vl_8b_int8_convrot.safetensors \
  "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-Quantized/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors"

# 3. VAE (bf16)
!wget -c -q --show-progress -O ComfyUI/models/vae/qwen_image_2.1_vae_bf16.safetensors \
  "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-Quantized/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors"

!ls -lh ComfyUI/models/unet/ ComfyUI/models/clip/ ComfyUI/models/vae/


In [ ]:
# Launch 8 ComfyUI instances (one on each TPU core)
import sys
sys.path.insert(0, "ComfyUI")
from tools.tpu_orchestrator import TPUClusterManager

manager = TPUClusterManager(cores=8, base_port=8188, comfy_root="ComfyUI")
manager.start(log_dir="ComfyUI/logs/tpu_cluster")
ready = manager.wait_ready(timeout=180)
print(f"Cluster initialized and ready: {ready}")


In [ ]:
# Dispatch 8 parallel prompts across all 8 TPU cores simultaneously
import json
import time

with open("ComfyUI/workflows/qwen_image_tpu_test.json", "r") as f:
    wf = json.load(f)
prompt_graph = wf.get("prompt", wf)

subjects = [
    "nebula galaxy exploding with celestial turquoise dust",
    "cyberpunk cityscape at night with glowing neon reflections",
    "ancient mystical temple hidden in lush mountain rainforest",
    "futuristic solar sailing ship skimming atmospheric plasma",
    "bioluminescent coral reef teeming with radiant aquatic life",
    "crystal ice fortress beneath a shimmering aurora borealis",
    "enchanted giant redwood forest illuminated by glowing spirits",
    "sleek retro-futuristic landspeeder drifting across desert dunes"
]

prompts = []
for i, subj in enumerate(subjects):
    p = json.loads(json.dumps(prompt_graph))
    p["4"]["inputs"]["text"] = f"{subj}, hyper-detailed digital art masterpiece 8k"
    p["7"]["inputs"]["seed"] = 1000 + i
    prompts.append(p)

print("Dispatching 8 parallel image generation requests across all 8 TPU cores...")
t0 = time.time()
results = manager.dispatch_parallel(prompts, timeout=600)
elapsed = time.time() - t0

print(f"Parallel Batch Completed in {elapsed:.2f} seconds!")
print(f"Successful Prompts: {results['successful_prompts']} / {results['total_prompts']}")
print(f"Throughput: {results['throughput_images_per_min']} images/minute")


In [ ]:
# Export artifacts to /kaggle/working/output for CLI download
import shutil

output_target = "/kaggle/working/output"
os.makedirs(output_target, exist_ok=True)

with open(os.path.join(output_target, "benchmark_report.json"), "w") as f:
    json.dump(results, f, indent=2)

comfy_out = "ComfyUI/output"
if os.path.exists(comfy_out):
    for fname in os.listdir(comfy_out):
        if fname.endswith((".png", ".jpg", ".webp")):
            shutil.copy(os.path.join(comfy_out, fname), os.path.join(output_target, fname))

!ls -lh /kaggle/working/output/

manager.stop()
print("All 8 TPU ComfyUI instances cleanly shut down.")
